# CardioIA – Fase 2 | Parte 2
## Classificador de Risco Cardíaco por Processamento de Linguagem Natural

**Grupo 69 – FIAP | Inteligência Artificial aplicada à Saúde**

---

### Contexto e Continuidade com a Parte 1

Na **Parte 1**, construímos um sistema de diagnóstico baseado em regras simbólicas: buscávamos termos do `mapa_conhecimento.csv` dentro das frases dos pacientes por correspondência exata de substring. Essa abordagem apresentava limitações conhecidas — por exemplo, o termo curto `"dor"` poderia gerar falsos positivos ao ser encontrado dentro de outras palavras, e o sistema não conseguia aprender novos padrões automaticamente.

Na **Parte 2**, substituímos essa lógica simbólica por um **modelo estatístico supervisionado** que aprende padrões diretamente das frases. Utilizamos TF-IDF para transformar texto em vetores numéricos e treinamos classificadores de Machine Learning para prever se uma frase de sintomas representa **alto risco** ou **baixo risco** cardiovascular.

Essa evolução reproduz, em escala acadêmica, a lógica por trás de **sistemas reais de triagem clínica automatizada**.

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Configuração visual
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Importações concluídas com sucesso.')

## 2. Carregamento e Análise Exploratória (EDA)

O dataset `frases_risco.csv` foi construído manualmente com frases em português simulando relatos reais de pacientes em triagem clínica. Cada frase está rotulada como **`alto risco`** ou **`baixo risco`**.

> **Nota contextual:** O dataset estruturado da Fase 1 (`dados_cardiacos.csv`) contém 500 registros clínicos com 4 classes de risco (Baixo, Moderado, Alto, Muito Alto), confirmando que a triagem multi-nível é um problema real neste domínio. Para esta etapa, simplificamos para classificação binária, que é o paradigma mais utilizado em sistemas de triagem automatizada de urgência.

In [ ]:
df = pd.read_csv('../dataset/frases_risco.csv', encoding='utf-8')

print(f'Dataset carregado: {df.shape[0]} frases | {df.shape[1]} colunas')
print(f'Colunas: {list(df.columns)}')
print()
df.head(6)

In [ ]:
# Distribuição das classes
contagem = df['situacao'].value_counts()
print('Distribuição das classes:')
print(contagem.to_string())
print(f'\nBalanceamento: {contagem.min()/contagem.max():.1%} (razão min/max)')

fig, ax = plt.subplots(figsize=(6, 4))
cores = ['#d62728', '#2ca02c']
contagem.plot(kind='bar', ax=ax, color=cores, edgecolor='white', width=0.5)
ax.set_title('Distribuição das Classes de Risco', fontsize=13, fontweight='bold')
ax.set_xlabel('Classe', fontsize=11)
ax.set_ylabel('Quantidade de Frases', fontsize=11)
ax.set_xticklabels(contagem.index, rotation=0)
for bar, valor in zip(ax.patches, contagem.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(valor), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Exemplos de frases por classe
print('=' * 65)
print('EXEMPLOS DE FRASES — ALTO RISCO')
print('=' * 65)
for frase in df[df['situacao'] == 'alto risco']['frase'].sample(3, random_state=42):
    print(f'  • {frase}')

print()
print('=' * 65)
print('EXEMPLOS DE FRASES — BAIXO RISCO')
print('=' * 65)
for frase in df[df['situacao'] == 'baixo risco']['frase'].sample(3, random_state=42):
    print(f'  • {frase}')

In [ ]:
# Análise do comprimento das frases
df['num_palavras'] = df['frase'].apply(lambda x: len(x.split()))

print('Estatísticas do comprimento das frases (número de palavras):')
print(df.groupby('situacao')['num_palavras'].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
for situacao, cor, label in [('alto risco', '#d62728', 'Alto Risco'),
                               ('baixo risco', '#2ca02c', 'Baixo Risco')]:
    subset = df[df['situacao'] == situacao]['num_palavras']
    ax.hist(subset, bins=10, alpha=0.6, color=cor, label=label, edgecolor='white')
ax.set_title('Distribuição do Comprimento das Frases', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Palavras', fontsize=11)
ax.set_ylabel('Frequência', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Pré-processamento e Vetorização com TF-IDF

**TF-IDF** (_Term Frequency – Inverse Document Frequency_) transforma frases de texto em vetores numéricos ponderando cada termo pela sua frequência no documento e pela raridade no corpus. Termos muito comuns em todas as frases (como "sinto", "tenho") recebem peso baixo; termos discriminativos (como "irradia", "síncope", "rotina") recebem peso alto.

**Justificativa dos hiperparâmetros escolhidos:**

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `max_features` | 200 | Com 140 documentos, vocabulário acima de 200 termos tende ao overfitting |
| `ngram_range` | (1, 2) | Bigramas são essenciais: "dor no peito" é muito mais informativo que "dor" isolado |
| `min_df` | 2 | Remove termos que aparecem em apenas 1 documento (sem poder estatístico) |
| `max_df` | 0.90 | Remove termos quase universais que não discriminam classes |
| `sublinear_tf` | True | Aplica log(1+tf) em vez de tf bruto, adequado para frases curtas |
| `strip_accents` | 'unicode' | Normaliza acentuação portuguesa para consistência |

> **Nota importante:** Aplicamos `fit_transform` **apenas no conjunto de treino** e `transform` no conjunto de teste, evitando **data leakage** (vazamento de informação do teste para o treino).

In [ ]:
X = df['frase']
y = df['situacao']

# Divisão treino/teste: 80%/20% estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Treino: {len(X_train)} frases ({y_train.value_counts().to_dict()})')
print(f'Teste:  {len(X_test)} frases ({y_test.value_counts().to_dict()})')

In [ ]:
# Vetorização TF-IDF
vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True,
    strip_accents='unicode'
)

# Fit APENAS no treino para evitar data leakage
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Dimensão da matriz de treino: {X_train_tfidf.shape}')
print(f'Dimensão da matriz de teste:  {X_test_tfidf.shape}')
print(f'Vocabulário TF-IDF: {X_train_tfidf.shape[1]} termos')
print(f'\nAmostra do vocabulário aprendido:')
vocab_amostra = list(vectorizer.get_feature_names_out()[:20])
print(' | '.join(vocab_amostra))

## 4. Treinamento dos Classificadores

Treinamos dois modelos para comparação:

- **Decision Tree (Árvore de Decisão):** Modelo interpretável que cria regras de decisão em forma de árvore. Tende a overfitting em datasets pequenos com alta dimensionalidade (como o TF-IDF).
- **Logistic Regression (Regressão Logística):** Modelo linear com regularização L2. Generaliza melhor em matrizes esparsas de alta dimensão, como as produzidas pelo TF-IDF.

In [ ]:
# Modelo 1: Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_tfidf, y_train)
y_pred_dt = dt.predict(X_test_tfidf)

acc_dt     = accuracy_score(y_test, y_pred_dt)
recall_dt  = recall_score(y_test, y_pred_dt, pos_label='alto risco')

print(f'Decision Tree')
print(f'  Acurácia:              {acc_dt:.2%}')
print(f'  Recall (alto risco):   {recall_dt:.2%}')

In [ ]:
# Modelo 2: Logistic Regression
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

acc_lr     = accuracy_score(y_test, y_pred_lr)
recall_lr  = recall_score(y_test, y_pred_lr, pos_label='alto risco')

print(f'Logistic Regression')
print(f'  Acurácia:              {acc_lr:.2%}')
print(f'  Recall (alto risco):   {recall_lr:.2%}')

In [ ]:
# Tabela comparativa dos modelos
resultados = pd.DataFrame({
    'Modelo': ['Decision Tree', 'Logistic Regression'],
    'Acurácia': [f'{acc_dt:.2%}', f'{acc_lr:.2%}'],
    'Recall (alto risco)': [f'{recall_dt:.2%}', f'{recall_lr:.2%}']
})

print('=== Comparação dos Modelos ===')
print(resultados.to_string(index=False))
print()
print('-> Modelo selecionado: Logistic Regression')
print('   Motivo: melhor generalização em matrizes TF-IDF esparsas e maior')
print('   recall para "alto risco", a métrica mais crítica em triagem médica.')

## 5. Avaliação do Modelo Selecionado (Regressão Logística)

**Por que o recall para "alto risco" é a métrica mais importante?**

Em triagem clínica, o custo de um **falso negativo** (classificar como "baixo risco" um paciente que é "alto risco") é muito maior do que o custo de um **falso positivo** (encaminhar desnecessariamente para avaliação urgente um paciente de baixo risco). Por isso, maximizar o **recall para "alto risco"** é uma decisão ética deliberada, não uma deficiência do modelo.

In [ ]:
# Matriz de Confusão
classes_ordem = ['alto risco', 'baixo risco']
cm = confusion_matrix(y_test, y_pred_lr, labels=classes_ordem)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Alto Risco', 'Baixo Risco']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusão – Logistic Regression', fontsize=13, fontweight='bold')
ax.set_xlabel('Classe Prevista', fontsize=11)
ax.set_ylabel('Classe Real', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('Interpretação:')
print(f'  Verdadeiro Positivo (Alto Risco correto):  {cm[0,0]}')
print(f'  Falso Negativo (Alto Risco como Baixo):    {cm[0,1]}  <- mais crítico')
print(f'  Falso Positivo (Baixo Risco como Alto):    {cm[1,0]}')
print(f'  Verdadeiro Negativo (Baixo Risco correto): {cm[1,1]}')

In [ ]:
# Relatório de Classificação completo
print('Relatório de Classificação — Logistic Regression')
print('=' * 55)
print(classification_report(
    y_test, y_pred_lr,
    target_names=['Alto Risco', 'Baixo Risco']
))
print(f'Acurácia geral: {acc_lr:.2%}')
print()
print('Nota: Com apenas 28 amostras no conjunto de teste, cada erro de')
print('classificação altera a acurácia em ~3,5 pontos percentuais.')
print('Os resultados devem ser interpretados com cautela nessa escala.')

In [ ]:
# Termos mais discriminativos aprendidos pelo modelo
feature_names = vectorizer.get_feature_names_out()
coefs = lr.coef_[0]

top_n = 12
indices_alto  = coefs.argsort()[-top_n:][::-1]
indices_baixo = coefs.argsort()[:top_n]

termos_alto  = [(feature_names[i], round(coefs[i], 3)) for i in indices_alto]
termos_baixo = [(feature_names[i], round(coefs[i], 3)) for i in indices_baixo]

print('Termos mais associados a ALTO RISCO (coeficiente positivo):')
for termo, coef in termos_alto:
    print(f'  {termo:<30} coef: {coef:+.3f}')

print()
print('Termos mais associados a BAIXO RISCO (coeficiente negativo):')
for termo, coef in termos_baixo:
    print(f'  {termo:<30} coef: {coef:+.3f}')

In [ ]:
# Visualização dos termos discriminativos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alto risco
ax = axes[0]
nomes_alto = [t[0] for t in termos_alto]
valores_alto = [t[1] for t in termos_alto]
y_pos = range(len(nomes_alto))
bars = ax.barh(y_pos, valores_alto, color='#d62728', alpha=0.8, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(nomes_alto, fontsize=9)
ax.set_title('Top Termos → ALTO RISCO', fontsize=12, fontweight='bold', color='#d62728')
ax.set_xlabel('Coeficiente TF-IDF')
ax.invert_yaxis()

# Baixo risco
ax = axes[1]
nomes_baixo = [t[0] for t in termos_baixo]
valores_baixo = [t[1] for t in termos_baixo]
y_pos = range(len(nomes_baixo))
bars = ax.barh(y_pos, [abs(v) for v in valores_baixo], color='#2ca02c', alpha=0.8, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(nomes_baixo, fontsize=9)
ax.set_title('Top Termos → BAIXO RISCO', fontsize=12, fontweight='bold', color='#2ca02c')
ax.set_xlabel('|Coeficiente TF-IDF|')
ax.invert_yaxis()

plt.suptitle('Termos mais Discriminativos por Classe', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Teste com frases novas (não vistas pelo modelo)
frases_novas = [
    "Sinto uma dor forte no peito que irradia para o braço esquerdo e estou suando frio.",
    "Fui ao médico para consulta de rotina e estou me sentindo muito bem.",
    "Meu coração acelerou muito, estou com falta de ar intensa e quase desmaiei.",
    "Tenho um cansaço leve ao final do dia que melhora com uma boa noite de sono.",
    "Desmaiei repentinamente e quando acordei não lembrava o que havia acontecido.",
    "Sinto uma leve dor muscular nas costas depois de carregar peso pesado ontem."
]

X_novas = vectorizer.transform(frases_novas)
previsoes = lr.predict(X_novas)
probabilidades = lr.predict_proba(X_novas)
classes = lr.classes_

print('Classificação de Frases Novas (não vistas no treino):')
print('=' * 70)
for frase, prev, prob in zip(frases_novas, previsoes, probabilidades):
    idx_alto = list(classes).index('alto risco')
    conf = prob[idx_alto] if prev == 'alto risco' else prob[1 - idx_alto]
    emoji = '🔴' if prev == 'alto risco' else '🟢'
    print(f'{emoji} [{prev.upper()}] (confiança: {conf:.0%})')
    print(f'   "{frase[:80]}"')
    print()

## 6. Considerações sobre Governança de Dados e Viés em IA

Esta seção reflete sobre as limitações, riscos e responsabilidades associados ao uso de sistemas de classificação automatizada em contextos médicos. Assim como na Parte 1, reconhecemos que a transparência sobre as limitações do sistema é tão importante quanto o seu desempenho técnico.

---

### 6.1 Origem e Representatividade dos Dados Simulados

O corpus `frases_risco.csv` foi construído **manualmente** para fins acadêmicos. As frases foram elaboradas com base em conhecimento médico geral, mas **não refletem a distribuição epidemiológica real** da população brasileira. Em um sistema real:
- A proporção de casos de alto risco seria muito menor (menos de 10% das triagens de emergência)
- Vocabulário regional e dialetal estaria presente (ex: termos diferentes no Nordeste e no Sul)
- Pacientes com baixa escolaridade usariam linguagem mais coloquial e menos estruturada

Modelos treinados neste corpus podem **não generalizar** para frases de pacientes reais.

---

### 6.2 Viés de Cobertura Demográfica

O corpus não representa adequadamente variações por:
- **Idade:** Idosos frequentemente descrevem sintomas de forma atípica (ex: infarto sem dor típica em mulheres acima de 65 anos)
- **Gênero:** A apresentação de sintomas cardíacos é diferente entre homens e mulheres — o modelo pode apresentar desempenho inferior para sintomas femininos
- **Nível socioeconômico:** Acesso desigual ao vocabulário médico influencia como os sintomas são relatados
- **Comorbidades:** Pacientes diabéticos podem ter sintomas de infarto sem dor torácica

---

### 6.3 A Escolha Ética: Recall vs. Precision

Na avaliação do modelo, priorizamos o **recall para "alto risco"** em detrimento da precisão. Esta é uma **decisão ética deliberada**:

| Tipo de Erro | Consequência Médica | Gravidade |
|-------------|--------------------|-----------|
| **Falso Negativo** (alto risco → baixo risco) | Paciente crítico não recebe atendimento urgente | **Risco de morte** |
| **Falso Positivo** (baixo risco → alto risco) | Paciente estável recebe avaliação desnecessária | Custo e inconveniente |

Em sistemas reais de triagem, essa assimetria de custos justifica aceitar mais falsos positivos para minimizar falsos negativos.

---

### 6.4 Limitações Fundamentais do TF-IDF

O TF-IDF trata texto como **"saco de palavras"** (_bag of words_), ignorando:

- **Negação:** `"não sinto dor no peito"` e `"sinto dor no peito"` têm vetores muito similares
- **Contexto:** `"dor leve"` e `"dor intensa"` diferem apenas em um modificador
- **Ordem:** A sequência das palavras é irrelevante para o TF-IDF
- **Semântica:** Sinônimos são tratados como termos completamente diferentes

Modelos de linguagem como **BERT** ou **BioPortuguese-BERT** resolveriam essas limitações ao codificar o contexto completo de cada palavra.

---

### 6.5 Recomendações para Evoluções Futuras

1. **Validação clínica:** O modelo deve ser avaliado por médicos emergencistas antes de qualquer aplicação real
2. **Corpus real anonimizado:** Substituir frases simuladas por dados reais de prontuários eletrônicos (com aprovação de comitê de ética)
3. **Cross-validation:** Substituir a divisão treino/teste única por validação cruzada com k≥5 folds para estimativas mais robustas
4. **Modelo multimodal:** Combinar texto (frases) com dados estruturados (pressão arterial, frequência cardíaca, idade) do `dados_cardiacos.csv`
5. **Monitoramento de drift:** Em produção, monitorar como o vocabulário dos pacientes muda ao longo do tempo
6. **Explicabilidade (XAI):** Implementar LIME ou SHAP para que médicos entendam por que cada classificação foi feita

---

> **Conclusão:** Este experimento reproduz com ferramentas acessíveis a lógica por trás de sistemas reais de triagem clínica. A acurácia técnica alcançada demonstra o potencial da abordagem, mas a responsabilidade ética e a qualidade dos dados continuam sendo os fatores mais críticos para viabilizar aplicações médicas seguras.